# Vision-Language Models in Apps

**Phase 10** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-10/01-vision-language-models.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

A product team has a working text-only Claude integration powering a support chat. The PM wants to add "analyze this screenshot" so users can upload an image and ask what is wrong. The engineering team opens the Anthropic docs and immediately hits four questions they cannot answer:

1. How do images actually get sent to the API? The messages format accepts strings, not file objects.
2. What does an image cost in tokens? They are already close to budget limits.
3. What file types and sizes are accepted? Users will upload whatever their phone produces.
4. What prompting strategies work for visual tasks? The zero-shot pattern they use for text does not feel right for images.

The team knows vis...

## The Concept

### How images travel through the API

The Anthropic Messages API uses a `content` array instead of a single `content` string. Each element in the array is a block: either a text block or an image block. That is the only structural change from a text-only request. The model receives both blocks together and reasons across them.

Images reach the API in one of two ways:

- **Base64 encoding**: the image bytes are encoded as a base64 string and sent directly in the request body. Required for images that are not publicly accessible via URL.
- **URL reference**: a publicly reachable HTTPS URL. The...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 10-01: Vision-Language Models in Apps
Sends an image to Claude and returns structured analysis as JSON.
Demo mode generates a synthetic PNG so no external image file is required.

Usage:
    python main.py                    # demo mode (generates synthetic PNG)
    python main.py sample.png         # analyze a local image file
"""

import anthropic
import base64
import json
import math
import struct
import sys
import zlib
from pathlib import Path


# --------------------------------------------------------------------------- #
# Demo image generator (stdlib only, no Pillow required)                      #
# --------------------------------------------------------------------------- #

def _make_minimal_png(width: int = 64, height: int = 64) -> bytes:
    """Generate a minimal valid PNG in memory using only stdlib."""

    def png_chunk(chunk_type: bytes, data: bytes) -> bytes:
        length = len(data)
        crc = zlib.crc32(chunk_type + data) & 0xFFFFFFFF
        return (
            struct.pack(">I", length)
            + chunk_type
            + data
            + struct.pack(">I", crc)
        )

    # IHDR: width, height, bit depth=8, color type=2 (RGB), compression=0, filter=0, interlace=0
    ihdr_data = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)

    # Build raw image data: each row has a filter byte (0 = None) + RGB pixels
    rows = bytearray()
    for y in range(height):
        rows.append(0)  # filter byte
        for x in range(width):
            rows.append(int(x * 255 / max(width - 1, 1)))   # R
            rows.append(int(y * 255 / max(height - 1, 1)))  # G
            rows.append(128)                                  # B

    compressed = zlib.compress(bytes(rows))

    png = (
        b"\x89PNG\r\n\x1a\n"
        + png_chunk(b"IHDR", ihdr_data)
        + png_chunk(b"IDAT", compressed)
        + png_chunk(b"IEND", b"")
    )
    return png


# --------------------------------------------------------------------------- #
# Token cost formula                                                           #
# --------------------------------------------------------------------------- #

def estimate_vision_tokens(width: int, height: int) -> int:
    """
    Anthropic vision token formula.
    Divides image into 32x32 tiles, charges 65 tokens per tile.
    """
    tiles_wide = math.ceil(width / 32)
    tiles_tall = math.ceil(height / 32)
    return tiles_wide * tiles_tall * 65


# --------------------------------------------------------------------------- #
# Core vision function                                                         #
# --------------------------------------------------------------------------- #

def analyze_image(
    image_bytes: bytes,
    media_type: str = "image/png",
    prompt: str = (
        "Analyze this image. Return JSON with keys: "
        "description (string), dominant_colors (list of strings), "
        "detected_text (list of strings), notable_elements (list of strings)."
    ),
    model: str = "claude-3-5-haiku-20241022",
) -> dict:
    """
    Send an image to Claude as a base64-encoded block.
    Returns the parsed JSON response from the model.
    """
    client = anthropic.Anthropic()

    b64_image = base64.standard_b64encode(image_bytes).decode("utf-8")

    print(f"  Image size: {len(image_bytes):,} bytes")

    message = client.messages.create(
        model=model,
        max_tokens=512,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": b64_image,
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt,
                    },
                ],
            }
        ],
    )

    raw_text = message.content[0].text

    # Strip markdown code fences if present
    if "```" in raw_text:
        parts = raw_text.split("```")
        raw_text = parts[1].lstrip("json").strip()

    try:
        analysis = json.loads(raw_text)
    except json.JSONDecodeError:
        analysis = {"raw_response": raw_text}

    return {
        "analysis": analysis,
        "usage": {
            "input_tokens": message.usage.input_tokens,
            "output_tokens": message.usage.output_tokens,
        },
        "model": message.model,
    }


# --------------------------------------------------------------------------- #
# Main                                                                        #
# --------------------------------------------------------------------------- #

def main():
    print("=== Lesson 10-01: Vision-Language Models in Apps ===\n")

    # Show token cost comparisons
    print("--- Vision token cost by image size ---")
    for label, w, h in [
        ("256x256", 256, 256),
        ("512x512", 512, 512),
        ("768x768", 768, 768),
        ("1920x1080", 1920, 1080),
        ("3840x2160", 3840, 2160),
    ]:
        tokens = estimate_vision_tokens(w, h)
        cost = tokens * 0.00000025  # Haiku input: $0.25/1M tokens
        print(f"  {label:12s}: {tokens:>7,} tokens  (~${cost:.4f})")
    print()

    # Determine image source
    if len(sys.argv) > 1:
        image_path = Path(sys.argv[1])
        if not image_path.exists():
            print(f"Error: file not found: {image_path}")
            sys.exit(1)
        print(f"Using image file: {image_path}")
        image_bytes = image_path.read_bytes()
        media_type = "image/jpeg" if image_path.suffix.lower() in (".jpg", ".jpeg") else "image/png"
    else:
        print("No image file provided. Generating synthetic demo PNG (64x64 gradient).")
        image_bytes = _make_minimal_png(64, 64)
        media_type = "image/png"

    print()
    print("Sending image to Claude for structured analysis...")
    result = analyze_image(image_bytes, media_type=media_type)

    print("\n--- Analysis Result ---")
    print(json.dumps(result["analysis"], indent=2))
    print("\n--- Token Usage ---")
    print(f"  Input tokens:  {result['usage']['input_tokens']:,}")
    print(f"  Output tokens: {result['usage']['output_tokens']:,}")
    print(f"  Model:         {result['model']}")
    cost = result["usage"]["input_tokens"] * 0.00000025
    print(f"  Estimated input cost: ${cost:.6f}")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Which image source type is required for images that are not reachable via a public URL?**

- A. source type: url, pointing to a signed S3 URL that expires in 60 seconds
- B. source type: base64, encoding the image bytes as a base64 string in the request body
- C. source type: file, uploading via the Files API before every request
- D. source type: stream, sending raw bytes over a WebSocket

**2. What is the single most impactful change you can make to reduce vision API costs without losing answer quality?**

- A. Switch from JPEG to PNG to reduce image size
- B. Resize images to 768px on the longest edge before encoding
- C. Reduce max_tokens from 512 to 256
- D. Use claude-3-haiku instead of claude-3-5-haiku-20241022

**3. Which approach minimizes API cost for this multi-prompt pattern?**

- A. Base64-encode the image in each of the three separate requests
- B. Use the Files API to upload once and reference the file ID in all three requests
- C. Combine all three prompts into a single request to reuse the image encoding
- D. Cache the base64 string in Redis and reuse it across requests

**4. Why is this use case unsuitable for vision LLMs, and what is the better approach?**

- A. Vision models cannot see buttons; use an OCR library instead
- B. Vision models return pixel coordinates in a proprietary format that requires post-processing
- C. Vision models are unreliable at precise pixel-level spatial reasoning; use a UI testing framework with DOM access instead
- D. Vision models only support images up to 512x512, which is too small for most UI screenshots

**5. What is the minimal structural change to the messages format?**

- A. Add an 'image' key at the top level of the request alongside 'messages'
- B. Change 'content' from a string to an array of typed blocks: one image block and one text block
- C. Create a separate 'vision' endpoint request and merge the results
- D. Base64-encode the image and concatenate it to the content string with a separator

**6. Which investigation step should you do first?**

- A. Switch to a larger model immediately - the current model lacks capacity
- B. Examine the actual images in the failure cases to categorize why extraction failed
- C. Increase max_tokens to allow longer responses
- D. Add retry logic to re-send failed extractions three times

**7. Should you use a vision LLM for this counting task? Why or why not?**

- A. Yes; vision LLMs count objects with pixel-perfect accuracy for groups up to 100
- B. No; vision LLMs are unreliable at counting more than 8-10 distinct items and should not be used for precise counting tasks
- C. Yes, but only if you use chain-of-thought prompting to force the model to count systematically
- D. No; vision LLMs cannot process photos with more than 10 people due to resolution limits

_Answers are in `checks.json` in the lesson directory._